# RAGU Control Room

Centralized orchestration for all RAGU notebooks.  
Edit the **Configuration** cell below, then **Run All Cells**.

| Group | Notebook | Purpose |
|-------|----------|--------|
| A | `ragu_diagnostic.ipynb` | Table health check + optional sandbox refresh |
| B | `bareboned_ragu_new.ipynb` | nonKMX + KMX RAGU scoring |
| B | `ste_ragu_postintegration.ipynb` | STE RAGU scoring |
| C | `nonkmx_gl_diagnostics.ipynb` | nonKMX gross-loss diagnostics |
| C | `STE_gl_diagnostics.ipynb` | STE gross-loss diagnostics |
| C | `kmx_mtns_ragu.ipynb` | KMX MTN-level diagnostics |

In [1]:
# =============================================================================
# RAGU CONTROL ROOM CONFIGURATION
# =============================================================================

# -- Global Dates and Granularity -----------------------------------------------
GRANULARITY       = 'w'           # 'q' = quarterly, 'm' = monthly, 'w' = weekly
START_DATE        = '2026-01-01'
END_DATE          = None          # None = today
STE_START_DATE    = '2025-10-07'  # STE inception date (hardcoded in STE SQL files)

# -- Which Notebooks to Run -----------------------------------------------------
RUN_DIAGNOSTIC            = False  # True = run table health + optional sandbox refresh

RUN_BAREBONED_RAGU        = True  # nonKMX + KMX scoring
RUN_STE_RAGU              = True   # STE scoring

RUN_NONKMX_GL_DIAGNOSTICS = False   # nonKMX gross-loss diagnostics
RUN_KMX_MTNS_RAGU         = False   # KMX MTN-level diagnostics
RUN_STE_GL_DIAGNOSTICS    = False   # STE gross-loss diagnostics

# -- Query and Cache Control ----------------------------------------------------
RUN_EVERY_QUERY   = True   # True = hit Redshift for all queries; False = use pickles
UPDATE_TABLES     = False  # True = refresh sandbox tables (diagnostic notebook only)

# -- Execution Mode -------------------------------------------------------------
PARALLEL_ALL      = True   # True = launch ALL enabled notebooks at once
                            # False = groups: A (diagnostic) -> B (scoring) -> C (GL diag)

# -- Upload Flags ---------------------------------------------------------------
RUN_SANDBOX       = False   # Upload current results to sandbox
RUN_HISTORICAL    = False   # Append to historical table

# -- Baselines (shared source of truth) -----------------------------------------
BASELINES = {
    'AN':  {'ltv': 1.94, 'new_recovery_unadjusted': 0.58,  'apr': 0.25},
    'FRN': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54,  'apr': 0.25},
    'STG': {'ltv': 1.94, 'new_recovery_unadjusted': 0.60,  'apr': 0.25},
    'FLD': {'ltv': 1.45, 'new_recovery_unadjusted': 0.54,  'apr': 0.235},
    'ENT': {'ltv': 1.45, 'new_recovery_unadjusted': 0.55,  'apr': 0.235},
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58,  'apr': 0.25},
    'STE': {'ltv': 1.94, 'new_recovery_unadjusted': 0.557, 'apr': 0.25},
}

# -- Model Parameters -----------------------------------------------------------
MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

# -- KMX MTN Baselines (used by kmx_mtns_ragu only) ----------------------------
KMX_BASELINES = {
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
}
KMX_MODEL_PARAMS = {
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'kmx_loss_scale': 0.067,
}

# -- LOBs -----------------------------------------------------------------------
LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX']
NONKMX_LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT']
EXCLUDED_VINTAGES = {}

# -- Diagnostic Settings --------------------------------------------------------
DIAG_N_VINTAGES   = 6
KMX_MTN_MODELS    = [3.0, 3.1, 3.2, 4.1]

In [2]:
# =============================================================================
# IMPORTS AND HELPERS
# =============================================================================
import papermill as pm
import time
import os
import glob
import pyodbc
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

NOTEBOOK_DIR = os.getcwd()
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, 'control_room_runs', RUN_TIMESTAMP)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Output directory: {OUTPUT_DIR}')


def run_notebook(notebook_name, params, label):
    """Execute a notebook via papermill, track timing, return status dict."""
    start = time.time()
    output_path = os.path.join(OUTPUT_DIR, notebook_name)
    try:
        pm.execute_notebook(
            os.path.join(NOTEBOOK_DIR, notebook_name),
            output_path,
            parameters=params,
            cwd=NOTEBOOK_DIR,
            log_output=True,
        )
        elapsed = round(time.time() - start, 1)
        return {'label': label, 'notebook': notebook_name, 'status': 'OK',
                'elapsed_sec': elapsed, 'output': output_path, 'error': None}
    except Exception as e:
        elapsed = round(time.time() - start, 1)
        return {'label': label, 'notebook': notebook_name, 'status': 'FAILED',
                'elapsed_sec': elapsed, 'output': output_path, 'error': str(e)[:500]}

Output directory: c:\Users\ahmed.ali\ragu-los\RAGU LOS\control_room_runs\20260625_094222


In [3]:
# =============================================================================
# PRE-FLIGHT CHECKS (informational, not blocking)
# =============================================================================

print('=' * 60)
print('PRE-FLIGHT CHECKS')
print('=' * 60)

# 1. Connection test
print('\n--- Connection Test ---')
try:
    with pyodbc.connect('DSN=Redshift_prod_new', timeout=15) as conn:
        conn.execute('SELECT 1')
    print('  Redshift connection: OK')
except Exception as e:
    print(f'  Redshift connection: FAILED -- {e}')

# 2. Cache freshness report
print('\n--- Cache Freshness ---')
cache_files = [
    '../../cache/ms_v1.pkl', '../../cache/ula_v1.pkl', '../../cache/dla_v1.pkl',
    '../../cache/new_recovery_v1.pkl', '../../cache/weekly_v1.pkl',
    '../../cache/ste_ula_v1.pkl', '../../cache/ste_recovery_v1.pkl', '../../cache/ste_weekly_v1.pkl',
    '../../cache/ula_kmx_v1.pkl',
]
cache_rows = []
for cf in cache_files:
    full = os.path.join(NOTEBOOK_DIR, cf)
    if os.path.exists(full):
        mtime = datetime.fromtimestamp(os.path.getmtime(full))
        size_mb = os.path.getsize(full) / (1024 * 1024)
        cache_rows.append({'file': cf, 'exists': True, 'modified': mtime.strftime('%Y-%m-%d %H:%M'), 'size_MB': round(size_mb, 1)})
    else:
        cache_rows.append({'file': cf, 'exists': False, 'modified': '-', 'size_MB': '-'})
cache_df = pd.DataFrame(cache_rows)
display(cache_df)

# 3. Excel lock detection (only checks files for enabled notebooks)
print('\n--- Excel Lock Check ---')
EXCEL_FILE_MAP = [
    ('../output/barebones_ragu.xlsx',        RUN_BAREBONED_RAGU,        'nonKMX+KMX Scoring'),
    ('../../ste/output/ste_ragu.xlsx',              RUN_STE_RAGU,              'STE Scoring'),
    ('../output/nonkmx_gl_diagnostics.xlsx', RUN_NONKMX_GL_DIAGNOSTICS, 'nonKMX GL Diag'),
    ('../../misc/STE_gl_diagnostics.xlsx',    RUN_STE_GL_DIAGNOSTICS,    'STE GL Diag'),
    ('../output/new_kmx_models.xlsx',        RUN_KMX_MTNS_RAGU,        'KMX MTN Diag'),
]
excel_rows = []
for ef, flag, label in EXCEL_FILE_MAP:
    if not flag:
        continue
    full = os.path.join(NOTEBOOK_DIR, ef)
    if not os.path.exists(full):
        excel_rows.append({'file': ef, 'notebook': label, 'status': 'OK (will be created)'})
    else:
        try:
            with open(full, 'a'):
                pass
            excel_rows.append({'file': ef, 'notebook': label, 'status': 'OK'})
        except PermissionError:
            excel_rows.append({'file': ef, 'notebook': label, 'status': 'LOCKED -- close in Excel'})

if excel_rows:
    excel_df = pd.DataFrame(excel_rows)
    def _excel_highlight(row):
        if 'LOCKED' in row['status']:
            return ['background-color: #dc3545; color: white; font-weight: bold'] * len(row)
        return ['background-color: #28a745; color: white; font-weight: bold'] * len(row)
    display(excel_df.style.apply(_excel_highlight, axis=1))
else:
    print('  No Excel files to check (no output notebooks enabled)')

# 4. Parameter validation
print('\n--- Parameter Validation ---')
errors = []
if GRANULARITY not in ('q', 'm', 'w'):
    errors.append(f'GRANULARITY={GRANULARITY!r} not in (q, m, w)')
try:
    pd.Timestamp(START_DATE)
except Exception:
    errors.append(f'START_DATE={START_DATE!r} is not a valid date')
if END_DATE is not None:
    try:
        pd.Timestamp(END_DATE)
    except Exception:
        errors.append(f'END_DATE={END_DATE!r} is not a valid date')
if not LOBS:
    errors.append('LOBS is empty')
if errors:
    for e in errors:
        print(f'  ERROR: {e}')
else:
    print('  All parameters valid')

print('\n' + '=' * 60)

PRE-FLIGHT CHECKS

--- Connection Test ---
  Redshift connection: OK

--- Cache Freshness ---


,file,exists,modified,size_MB
0,cache/ms_v1.pkl,True,2026-06-24 14:33,72.0
1,cache/ula_v1.pkl,True,2026-06-18 14:29,732.5
2,cache/dla_v1.pkl,True,2026-06-24 14:37,9.0
3,cache/new_recovery_v1.pkl,True,2026-06-24 14:38,96.6
4,cache/weekly_v1.pkl,True,2026-06-24 14:41,212.0
5,cache/ste_ula_v1.pkl,True,2026-06-18 11:24,29.4
6,cache/ste_recovery_v1.pkl,True,2026-06-18 11:24,1.3
7,cache/ste_weekly_v1.pkl,True,2026-06-18 11:24,3.6
8,cache/ula_kmx_v1.pkl,True,2026-05-27 09:21,101.5



--- Excel Lock Check ---


,file,notebook,status
0,barebones_ragu.xlsx,nonKMX+KMX Scoring,OK
1,ste_ragu.xlsx,STE Scoring,OK



--- Parameter Validation ---
  All parameters valid



In [4]:
# =============================================================================
# BUILD PARAMETER DICTS
# =============================================================================

diag_params = {
    'update_tables': UPDATE_TABLES,
}

bareboned_params = {
    'granularity': GRANULARITY,
    'START_DATE': START_DATE,
    'END_DATE': END_DATE,
    'run_every_query': RUN_EVERY_QUERY,
    'LOBS': LOBS,
    'BASELINES': {k: v for k, v in BASELINES.items() if k in LOBS},
    'MODEL_PARAMS': MODEL_PARAMS,
    'EXCLUDED_VINTAGES': EXCLUDED_VINTAGES,
    'run_sandbox': RUN_SANDBOX,
    'run_historical': RUN_HISTORICAL,
}

ste_params = {
    'granularity': GRANULARITY,
    'START_DATE': STE_START_DATE,
    'END_DATE': END_DATE,
    'run_every_query': RUN_EVERY_QUERY,
    'BASELINES': {'STE': BASELINES['STE']},
    'MODEL_PARAMS': MODEL_PARAMS,
    'run_sandbox': RUN_SANDBOX,
    'run_historical': RUN_HISTORICAL,
}

nonkmx_gl_params = {
    'granularity': GRANULARITY,
    'START_DATE': START_DATE,
    'END_DATE': END_DATE,
    'run_every_query': RUN_EVERY_QUERY,
    'LOBS': NONKMX_LOBS,
    'BASELINES': {k: v for k, v in BASELINES.items() if k in NONKMX_LOBS},
    'MODEL_PARAMS': {'mean_unit_loss': MODEL_PARAMS['mean_unit_loss'],
                     'unit_loss_to_model_score': MODEL_PARAMS['unit_loss_to_model_score']},
    'DIAG_N_VINTAGES': DIAG_N_VINTAGES,
}

ste_gl_params = {
    'granularity': GRANULARITY,
    'START_DATE': STE_START_DATE,
    'END_DATE': END_DATE,
    'run_every_query': RUN_EVERY_QUERY,
    'BASELINES': {'STE': BASELINES['STE']},
    'MODEL_PARAMS': {'mean_unit_loss': MODEL_PARAMS['mean_unit_loss'],
                     'unit_loss_to_model_score': MODEL_PARAMS['unit_loss_to_model_score']},
    'DIAG_N_VINTAGES': DIAG_N_VINTAGES,
}

kmx_params = {
    'granularity': GRANULARITY,
    'START_DATE': START_DATE,
    'END_DATE': END_DATE,
    'run_every_query': RUN_EVERY_QUERY,
    'BASELINES': KMX_BASELINES,
    'MODEL_PARAMS': KMX_MODEL_PARAMS,
    'MTN_MODELS': KMX_MTN_MODELS,
}

# -- Execution plan: (flag, notebook, params, label, group) ---------------------
NOTEBOOK_PLAN = [
    (RUN_DIAGNOSTIC,            'ragu_diagnostic.ipynb',         diag_params,      'Table Diagnostic',    'A'),
    (RUN_BAREBONED_RAGU,        'bareboned_ragu_new.ipynb',      bareboned_params, 'nonKMX+KMX Scoring',  'B'),
    (RUN_STE_RAGU,              'ste_ragu_postintegration.ipynb', ste_params,       'STE Scoring',         'B'),
    (RUN_NONKMX_GL_DIAGNOSTICS, 'nonkmx_gl_diagnostics.ipynb',   nonkmx_gl_params, 'nonKMX GL Diag',      'C'),
    (RUN_STE_GL_DIAGNOSTICS,    'STE_gl_diagnostics.ipynb',       ste_gl_params,    'STE GL Diag',         'C'),
    (RUN_KMX_MTNS_RAGU,         'kmx_mtns_ragu.ipynb',           kmx_params,       'KMX MTN Diag',        'C'),
]

print('Execution plan:')
for flag, nb, _, label, group in NOTEBOOK_PLAN:
    status = 'ENABLED' if flag else 'disabled'
    print(f'  Group {group} | {status:8s} | {label:20s} | {nb}')

Execution plan:
  Group A | disabled | Table Diagnostic     | ragu_diagnostic.ipynb
  Group B | ENABLED  | nonKMX+KMX Scoring   | bareboned_ragu_new.ipynb
  Group B | ENABLED  | STE Scoring          | ste_ragu_postintegration.ipynb
  Group C | disabled | nonKMX GL Diag       | nonkmx_gl_diagnostics.ipynb
  Group C | disabled | STE GL Diag          | STE_gl_diagnostics.ipynb
  Group C | disabled | KMX MTN Diag         | kmx_mtns_ragu.ipynb


In [5]:
# =============================================================================
# EXECUTE
# =============================================================================

enabled = [(nb, p, lbl, g) for flag, nb, p, lbl, g in NOTEBOOK_PLAN if flag]

if not enabled:
    print('No notebooks enabled. Set at least one RUN_* flag to True.')
else:
    results = []
    overall_start = time.time()

    if PARALLEL_ALL:
        print(f'=== PARALLEL_ALL: launching {len(enabled)} notebooks ===')
        with ThreadPoolExecutor(max_workers=len(enabled)) as pool:
            futures = {pool.submit(run_notebook, nb, p, lbl): lbl
                       for nb, p, lbl, g in enabled}
            for f in as_completed(futures):
                r = f.result()
                results.append(r)
                icon = 'OK' if r['status'] == 'OK' else 'FAILED'
                print(f"  [{icon}] {r['label']:25s}  ({r['elapsed_sec']}s)")
    else:
        for group_id in ['A', 'B', 'C']:
            group = [(nb, p, lbl) for nb, p, lbl, g in enabled if g == group_id]
            if not group:
                continue
            print(f"\n--- Group {group_id}: {', '.join(lbl for _, _, lbl in group)} ---")
            with ThreadPoolExecutor(max_workers=len(group)) as pool:
                futures = {pool.submit(run_notebook, nb, p, lbl): lbl
                           for nb, p, lbl in group}
                for f in as_completed(futures):
                    r = f.result()
                    results.append(r)
                    icon = 'OK' if r['status'] == 'OK' else 'FAILED'
                    print(f"  [{icon}] {r['label']:25s}  ({r['elapsed_sec']}s)")

    overall_elapsed = round(time.time() - overall_start, 1)
    print(f'\nAll done in {overall_elapsed}s')

Unable to parse line 10 'END_DATE = None  # None = auto-detect from today's date'.


=== PARALLEL_ALL: launching 2 notebooks ===


Unable to parse line 12 'END_DATE = None  # None = auto-detect from today's date'.
Unable to parse line 13 'run_every_query = True  # True = run SQL queries; False = use cached pickles'.
Unable to parse line 15 'run_every_query = True  # True = run SQL queries; False = use cached pickles'.
Passed unknown parameter: END_DATE
Passed unknown parameter: granularity
Passed unknown parameter: run_every_query
Passed unknown parameter: END_DATE
Passed unknown parameter: run_sandbox
Passed unknown parameter: run_every_query
Passed unknown parameter: run_historical
Passed unknown parameter: run_sandbox
Passed unknown parameter: run_historical


Executing:   0%|          | 0/14 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

  [OK] STE Scoring                (200.1s)
  [FAILED] nonKMX+KMX Scoring         (345.7s)

All done in 345.7s


In [6]:
# =============================================================================
# SUMMARY DASHBOARD
# =============================================================================

if 'results' in dir() and results:
    summary_df = pd.DataFrame(results)
    summary_df = summary_df[['label', 'notebook', 'status', 'elapsed_sec', 'output', 'error']]

    def _highlight(row):
        if row['status'] == 'OK':
            return ['background-color: #28a745; color: white; font-weight: bold'] * len(row)
        return ['background-color: #dc3545; color: white; font-weight: bold'] * len(row)

    display(summary_df.style.apply(_highlight, axis=1))

    ok = sum(1 for r in results if r['status'] == 'OK')
    fail = sum(1 for r in results if r['status'] == 'FAILED')
    print(f'\nResults: {ok} OK, {fail} FAILED out of {len(results)} notebooks')
    print(f'Output directory: {OUTPUT_DIR}')

    if fail > 0:
        print('\n--- FAILED NOTEBOOKS ---')
        for r in results:
            if r['status'] == 'FAILED':
                print(f"\n{r['label']} ({r['notebook']}):")
                print(f"  {r['error']}")
else:
    print('No results to display. Run the Execute cell first.')

,label,notebook,status,elapsed_sec,output,error
0,STE Scoring,ste_ragu_postintegration.ipynb,OK,200.100000,c:\Users\ahmed.ali\ragu-los\RAGU LOS\control_room_runs\20260625_094222\ste_ragu_postintegration.ipynb,nan
1,nonKMX+KMX Scoring,bareboned_ragu_new.ipynb,FAILED,345.700000,c:\Users\ahmed.ali\ragu-los\RAGU LOS\control_room_runs\20260625_094222\bareboned_ragu_new.ipynb,"--------------------------------------------------------------------------- Exception encountered at ""In [13]"": --------------------------------------------------------------------------- IncompatibleFrequency Traceback (most recent call last) File ~\.conda\envs\new1\Lib\site-packages\pandas\core\arrays\datetimelike.py:555, in DatetimeLikeArrayMixin._validate_comparison_value(self, other) 554 try: --> 555 self._check_compatible_with(other) 556 except TypeError as"



Results: 1 OK, 1 FAILED out of 2 notebooks
Output directory: c:\Users\ahmed.ali\ragu-los\RAGU LOS\control_room_runs\20260625_094222

--- FAILED NOTEBOOKS ---

nonKMX+KMX Scoring (bareboned_ragu_new.ipynb):
  
---------------------------------------------------------------------------
Exception encountered at "In [13]":
---------------------------------------------------------------------------
IncompatibleFrequency                     Traceback (most recent call last)
File ~\.conda\envs\new1\Lib\site-packages\pandas\core\arrays\datetimelike.py:555, in DatetimeLikeArrayMixin._validate_comparison_value(self, other)
    554 try:
--> 555     self._check_compatible_with(other)
    556 except TypeError as
